# 03 · Find factor-similar tickers (build a peer set)

> Goal: given one ticker's current factor state, find historical setups that
> looked similar. Useful for peer-grouping, regime exploration, or just
> sanity-checking "is this state unusual?"

## What we'll do

1. Pull AAPL's nearest historical analogues
2. Sort by similarity
3. Compare their factor values to AAPL's

In [1]:
# Setup — works with or without an API key.
# With FACTORWEAVE_API_KEY set, we use the full API (10,000+ tickers).
# Without one, we fall back to /demo/{ticker} (AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA, JPM).
import os, json
import requests

API_BASE = "https://factorweave.com/api"
API_KEY = os.environ.get("FACTORWEAVE_API_KEY")
DEMO_TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "JPM"]
MODE = "live" if API_KEY else "demo"
print(f"Running in {MODE} mode.", "Key prefix:", (API_KEY[:8] + '…') if API_KEY else "(none)")


def fw_demo(ticker: str) -> dict:
    """Demo endpoint — no auth, 8 sample tickers, current snapshot only."""
    r = requests.get(f"{API_BASE}/demo/{ticker}", timeout=10)
    r.raise_for_status()
    return r.json()


def fw_features(ticker: str, **kwargs) -> dict:
    """Authed features endpoint when a key is available; demo fallback otherwise."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/features/{ticker}",
                         params=kwargs,
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — reshape demo response to look like the authed one
    d = fw_demo(ticker)
    return {"rows": [{"ticker": d["ticker"], "date": d["as_of"], **d["factors"]}]}


def fw_top(factor: str, n: int = 25) -> dict:
    """Top-N by a factor. Needs auth for the full universe; in demo mode we
    rank the 8 demo tickers locally."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/top",
                         params={"factor": factor, "n": n},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — fetch each demo ticker, sort locally
    rows = []
    for t in DEMO_TICKERS:
        d = fw_demo(t)
        if factor in d["factors"]:
            rows.append({"ticker": t, "date": d["as_of"], factor: d["factors"][factor]})
    rows.sort(key=lambda r: r[factor], reverse=True)
    return {"rows": rows[:n]}


def fw_similar(ticker: str, method: str = "cosine", limit: int = 10) -> dict:
    """Similarity search. Demo endpoint includes pre-computed `similar` set."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/vector-search/similar/{ticker}",
                         params={"method": method, "limit": limit, "min_lookback_days": 30},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — uses the `similar` array baked into the demo response
    d = fw_demo(ticker)
    return {"ticker": ticker, "method": "cosine (demo)", "neighbors": d.get("similar", [])[:limit]}


def fw_market_context() -> dict:
    """Universe analytics. Public on FREE, fuller on HOBBY+."""
    headers = {"X-API-Key": API_KEY} if API_KEY else {}
    r = requests.get(f"{API_BASE}/market-context", params={"latest": 1}, headers=headers, timeout=10)
    if r.status_code == 401:
        return {"_note": "market-context requires auth in demo mode"}
    r.raise_for_status()
    return r.json()


Running in demo mode. Key prefix: (none)


## Find AAPL's 10 nearest neighbors

The `cosine` method finds historical (ticker, date) pairs whose factor
profile is closest to AAPL today — measured as cosine similarity of the
factor vector.

`min_lookback_days=30` filters out today's co-moving ETFs so you get
genuine historical analogues.

In [2]:
import pandas as pd

result = fw_similar("AAPL", method="cosine", limit=10)
print("method:", result["method"])
print()

neighbors = pd.DataFrame(result["neighbors"])
neighbors

method: cosine (demo)



,ticker,date,similarity,rsi,mom,ret_20d,comp_score
0,INSW,2025-08-04,0.7682,69.916766,0.032226,0.032226,-0.061540
1,FRME,2000-12-14,0.7595,41.176684,-0.035618,-0.035618,-0.088670
2,SMPL,2025-08-01,0.7589,16.887417,-0.062695,-0.062695,0.113357
3,FIX,2011-07-21,0.7491,58.411215,0.057859,0.057859,0.174163
4,LTCHW,2022-12-08,0.7481,48.094193,-0.320316,-0.320316,-4.085054
5,UTG,2011-11-07,0.7475,51.388889,0.023908,0.023908,0.150205
6,TNK,2025-08-04,0.7453,56.878520,0.004898,0.004898,-0.055947
7,XWEL,2011-01-14,0.7436,31.460674,-0.142241,-0.142241,0.005774


## Compare AAPL today vs its neighbors' average

In [3]:
aapl = fw_features("AAPL")["rows"][0]

compare_cols = [c for c in ("rsi", "mom", "ret_20d", "comp_score")
                if c in neighbors.columns and c in aapl]

if not compare_cols:
    print("Demo response is too sparse for this comparison — try with a key.")
else:
    rows = []
    for c in compare_cols:
        rows.append({
            "factor": c,
            "AAPL today": round(aapl[c], 4),
            "neighbors mean": round(neighbors[c].mean(), 4),
            "neighbors min": round(neighbors[c].min(), 4),
            "neighbors max": round(neighbors[c].max(), 4),
        })
    pd.DataFrame(rows)

## Why this is screening, not prediction

Each neighbor is a historical setup that *looked* like today's AAPL on the
factor axes we measure. That tells you something about the *shape* of
similar situations — what tickers, what regimes, what dates. It does not
tell you what AAPL will do next.

Our own leak-free testing shows the realized forward returns of these
neighbors do not predict AAPL's realized forward return (cross-sectional
IC is statistically zero). The one signal that *is* meaningful: their
forward realized **volatility** does correlate with AAPL's — risk-coherence.

For the methodology, see [factorweave.com/research.html](https://factorweave.com/research.html).

## Next

→ `04-leak-free-backtest.ipynb` — assemble a clean (features → forward-return) dataset for your own modeling.